# 7. Évaluation de l'assistant text-to-SQL

**Le problème :** une requête valide mais fausse (« 0 accident le samedi ») ne produit aucune erreur. C'est une **erreur silencieuse**.

**La solution :** comme les tests unitaires en développement, on prépare un **jeu de test**. Ce sont 20 questions dont on connaît la bonne réponse, grâce à une requête de **référence** écrite à la main.
Pour chaque question, on compare le résultat de l'assistant à celui de la référence, puis on calcule un **taux de bonnes réponses**.

Le code de l'assistant (notebook 06) a été regroupé dans le module `assistant.py`, à la racine du projet, pour pouvoir le réutiliser ici et dans le dashboard.

In [10]:
import json
import numbers
import sys
import time

import pandas as pd

sys.path.insert(0, "..")          # le module assistant.py est dans le dossier parent
from assistant import repondre, sql

# Le jeu de test est rangé dans un fichier séparé : on peut ajouter des questions sans toucher au code
with open("../evaluation/jeu_de_test.json", encoding="utf-8") as f:
    JEU_DE_TEST = json.load(f)

pd.DataFrame(JEU_DE_TEST)

,categorie,question,sql
0,Comptage simple,Combien de personnes ont été impliquées dans un accident ?,SELECT COUNT(*) FROM usagers
1,Comptage simple,Combien de personnes ont été tuées au total ?,SELECT COUNT(*) FROM usagers WHERE gravite = 'Tué'
2,Comptage simple,Combien de piétons ont été tués ?,SELECT COUNT(*) FROM usagers WHERE categorie = 'Piéton' AND gravite = 'Tué'
3,Comptage simple,Combien de femmes ont été tuées ?,SELECT COUNT(*) FROM usagers WHERE sexe_lib = 'Femme' AND gravite = 'Tué'
4,Comptage simple,Combien d'accidents ont eu lieu un samedi ?,SELECT COUNT(*) FROM accidents WHERE jour_semaine = 'Samedi'
5,Comptage simple,Combien d'accidents ont eu lieu en juillet ?,SELECT COUNT(*) FROM accidents WHERE mois = 7
6,Comptage simple,Combien d'accidents ont eu lieu avant 6 heures du matin ?,SELECT COUNT(*) FROM accidents WHERE heure < 6
7,Comptage simple,Combien d'accidents ont eu lieu par forte pluie ?,SELECT COUNT(*) FROM accidents WHERE meteo = 'Pluie forte'
8,Jointure,Combien de personnes ont été tuées à Paris ?,SELECT COUNT(*) FROM usagers AS u JOIN accidents AS a ON u.Num_Acc = a.Num_Acc WHERE u.gravite = 'Tué' AND a.dep = '75'
9,Jointure,Combien de piétons ont été tués la nuit ?,SELECT COUNT(*) FROM usagers AS u JOIN accidents AS a ON u.Num_Acc = a.Num_Acc WHERE u.categorie = 'Piéton' AND u.gravite = 'Tué' AND a.periode = 'Nuit'


## Comparer deux résultats

Deux résultats corrects peuvent avoir une forme différente : noms de colonnes, ordre des lignes, arrondis (27,98 contre 27,982…), ou une colonne en plus (l'assistant peut renvoyer `heure` **et** `nb_accidents` quand la référence ne renvoie que `heure`).

On compare donc les **valeurs**, colonne par colonne :
- même nombre de lignes ;
- chaque colonne de la référence doit se retrouver dans le résultat de l'assistant (valeurs triées, nombres arrondis à 1 décimale).

C'est une comparaison volontairement simple. Elle est un peu tolérante (elle ignore les noms et l'ordre des colonnes), mais elle détecte bien les résultats faux.

In [11]:
def normaliser(df):
    """Transforme un DataFrame en liste de colonnes : valeurs arrondies et triées."""
    colonnes = []
    for nom in df.columns:
        valeurs = [round(float(v), 1) if isinstance(v, numbers.Number) and not pd.isna(v) else v
                   for v in df[nom]]
        colonnes.append(sorted(valeurs, key=str))
    return colonnes


def meme_resultat(resultat, reference):
    if len(resultat) != len(reference):
        return False
    colonnes_resultat = normaliser(resultat)
    return all(colonne in colonnes_resultat for colonne in normaliser(reference))


# Petit test de la fonction elle-même
ref = pd.DataFrame({"heure": [17]})
print(meme_resultat(pd.DataFrame({"h": [17], "nb": [4607]}), ref))   # True : bonne valeur + colonne en plus
print(meme_resultat(pd.DataFrame({"h": [18]}), ref))                 # False : mauvaise valeur

True
False


## Évaluer un modèle sur tout le jeu de test

Pour chaque question, 4 issues possibles :
- ✅ **correct** : même résultat que la référence ;
- ❌ **faux** : la requête fonctionne, mais le résultat est différent (l'erreur silencieuse) ;
- 💥 **erreur SQL** : la requête est invalide ;
- 🛑 **refusé** : la requête est bloquée par le contrôle de sécurité.

On mesure aussi le **temps de réponse** avec `time.perf_counter()`.

In [12]:
def evaluer(modele):
    lignes = []
    for test in JEU_DE_TEST:
        debut = time.perf_counter()
        r = repondre(test["question"], modele=modele)
        duree = time.perf_counter() - debut

        if r["statut"] == "refus":
            statut = "🛑 refusé"
        elif r["statut"] == "erreur":
            statut = "💥 erreur SQL"
        elif meme_resultat(r["resultat"], sql(test["sql"])):
            statut = "✅ correct"
        else:
            statut = "❌ faux"

        lignes.append({"modele": modele, "categorie": test["categorie"], "question": test["question"],
                       "statut": statut, "duree_s": round(duree, 1), "sql_genere": r["sql"]})
        print(statut, "·", test["question"])
    return pd.DataFrame(lignes)

In [14]:
resultats_7b = evaluer("qwen2.5-coder:7b")

✅ correct · Combien de personnes ont été impliquées dans un accident ?
✅ correct · Combien de personnes ont été tuées au total ?
✅ correct · Combien de piétons ont été tués ?
✅ correct · Combien de femmes ont été tuées ?
✅ correct · Combien d'accidents ont eu lieu un samedi ?
✅ correct · Combien d'accidents ont eu lieu en juillet ?
✅ correct · Combien d'accidents ont eu lieu avant 6 heures du matin ?
✅ correct · Combien d'accidents ont eu lieu par forte pluie ?
❌ faux · Combien de personnes ont été tuées à Paris ?
✅ correct · Combien de piétons ont été tués la nuit ?
✅ correct · Combien de personnes ont été tuées hors agglomération par temps de pluie légère ?
✅ correct · Combien de personnes ont été tuées dans les Bouches-du-Rhône ?
✅ correct · Combien d'accidents ont eu lieu en Gironde ?
✅ correct · Quel est le nombre d'accidents pour chaque période de la journée ?
✅ correct · Quels sont les 3 départements avec le plus d'accidents ?
✅ correct · Quelle heure de la journée compte le plu

## Le score

In [15]:
def bilan(resultats):
    print(f"Taux de bonnes réponses : {(resultats['statut'] == '✅ correct').mean():.0%}")
    print(f"Temps moyen par question : {resultats['duree_s'].mean():.1f} s\n")
    # Répartition des issues par catégorie de question
    return pd.crosstab(resultats["categorie"], resultats["statut"], margins=True, margins_name="Total")


bilan(resultats_7b)

Taux de bonnes réponses : 95%
Temps moyen par question : 12.1 s



statut,✅ correct,❌ faux,Total
categorie,,,
Comptage simple,9,0,9
Jointure,3,1,4
Moyenne,2,0,2
Pourcentage,3,0,3
Regroupement,4,0,4
Total,21,1,22


## Analyser les échecs

Le score ne suffit pas : il faut **comprendre** chaque échec. Valeur mal écrite ? Mauvaise table ? Mauvais dénominateur ?
C'est cette analyse qui permet d'améliorer le prompt.

In [16]:
pd.set_option("display.max_colwidth", None)     # affiche le SQL en entier
resultats_7b[resultats_7b["statut"] != "✅ correct"][["question", "statut", "sql_genere"]]

,question,statut,sql_genere
8,Combien de personnes ont été tuées à Paris ?,❌ faux,SELECT COUNT(*) AS nb_tues FROM usagers AS u JOIN accidents AS a ON u.Num_Acc = a.Num_Acc WHERE u.gravite = 'Tué' AND a.com = 'Paris'


## Comparaison avec le petit modèle

Le même jeu de test, avec `qwen2.5:3b`. C'est l'intérêt d'un jeu de test : **comparer objectivement** deux modèles, ou deux versions d'un prompt.

In [17]:
resultats_3b = evaluer("qwen2.5:3b")

✅ correct · Combien de personnes ont été impliquées dans un accident ?
✅ correct · Combien de personnes ont été tuées au total ?
❌ faux · Combien de piétons ont été tués ?
✅ correct · Combien de femmes ont été tuées ?
✅ correct · Combien d'accidents ont eu lieu un samedi ?
❌ faux · Combien d'accidents ont eu lieu en juillet ?
✅ correct · Combien d'accidents ont eu lieu avant 6 heures du matin ?
✅ correct · Combien d'accidents ont eu lieu par forte pluie ?
✅ correct · Combien de personnes ont été tuées à Paris ?
💥 erreur SQL · Combien de piétons ont été tués la nuit ?
✅ correct · Combien de personnes ont été tuées hors agglomération par temps de pluie légère ?
💥 erreur SQL · Combien de personnes ont été tuées dans les Bouches-du-Rhône ?
✅ correct · Combien d'accidents ont eu lieu en Gironde ?
✅ correct · Quel est le nombre d'accidents pour chaque période de la journée ?
✅ correct · Quels sont les 3 départements avec le plus d'accidents ?
✅ correct · Quelle heure de la journée compte le 

In [18]:
comparaison = pd.concat([resultats_3b, resultats_7b])
comparaison.groupby("modele").agg(
    taux_correct=("statut", lambda s: f"{(s == '✅ correct').mean():.0%}"),
    temps_moyen_s=("duree_s", "mean"),
).round(1)

,taux_correct,temps_moyen_s
modele,,
qwen2.5-coder:7b,95%,12.1
qwen2.5:3b,68%,8.8
